In [35]:
import os
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv


In [36]:
load_dotenv()

True

In [37]:
load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    raise ValueError("GOOGLE_API_KEY is not set. Add it to your environment or .env file before running this notebook.")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)


In [38]:
# create a state

class LLMState(TypedDict):

    question: str
    answer: str

In [39]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = llm.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state


In [40]:
# create our graph

workflow = StateGraph(LLMState)

# add nodes
workflow.add_node('llm_qa', llm_qa)

# add edges
workflow.add_edge(START, 'llm_qa')
workflow.add_edge('llm_qa', END)

# compile
workflow = workflow.compile()


In [41]:
# execute
initial_state = {'question': 'How far is moon from the earth?'}
final_state = workflow.invoke(initial_state)
print(final_state['answer'])


The average distance from the Earth to the Moon is approximately 384,400 kilometers (238,900 miles). This distance is constantly changing due to the elliptical shape of the Moon's orbit around the Earth. At its closest point (called perigee), the Moon is about 363,300 kilometers (225,300 miles) away, and at its farthest point (apogee), it is about 405,500 kilometers (252,000 miles) away.


In [42]:
llm.invoke('How far is moon from the earth?').content


"The average distance from the Earth to the Moon is approximately 384,400 kilometers (238,900 miles). This distance is constantly changing due to the elliptical shape of the Moon's orbit around the Earth. At its closest point (called perigee), the Moon is about 363,300 kilometers (225,300 miles) away, and at its farthest point (apogee), it is about 405,500 kilometers (252,000 miles) away."